In [29]:
from airflow.sdk import dag, task
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param
from airflow.sdk.definitions.param import ParamsDict
from sqlalchemy import create_engine, VARCHAR, Integer,Date, inspect
import os
import pandas as pd
from datetime import datetime
import zipfile
import numpy as np
import requests
from datetime import datetime
from pathlib import Path
import shutil
import boto3
from botocore.config import Config


In [114]:
def candidato(file_csv):
    colunas_leitura = ['dt_geracao', 'hh_geracao', 'ano_eleicao', 'cd_tipo_eleicao',
       'nm_tipo_eleicao', 'nr_turno', 'cd_eleicao', 'ds_eleicao', 'dt_eleicao',
       'tp_abrangencia', 'sg_uf', 'sg_ue', 'nm_ue', 'cd_cargo', 'ds_cargo',
       'sq_candidato', 'nr_candidato', 'nm_candidato', 'nm_urna_candidato',
       'nm_social_candidato', 'nr_cpf_candidato', 'ds_email',
       'cd_situacao_candidatura', 'ds_situacao_candidatura', 'tp_agremiacao',
       'nr_partido', 'sg_partido', 'nm_partido', 'nr_federacao',
       'nm_federacao', 'sg_federacao', 'ds_composicao_federacao',
       'sq_coligacao', 'nm_coligacao', 'ds_composicao_coligacao',
       'sg_uf_nascimento', 'dt_nascimento', 'nr_titulo_eleitoral_candidato',
       'cd_genero', 'ds_genero', 'cd_grau_instrucao', 'ds_grau_instrucao',
       'cd_estado_civil', 'ds_estado_civil', 'cd_cor_raca', 'ds_cor_raca',
       'cd_ocupacao', 'ds_ocupacao', 'cd_sit_tot_turno', 'ds_sit_tot_turno',
       'id_candidato', 'nome_foto', 'url_foto']
    dfs = []
    lista_links = []
    # Configurações do Cloudflare
    account_id = '8e2ebba9301e868cfdedc13086d6c140'
    access_key = 'a35b4c037c701ade40b95d0c48e4b4b3'
    secret_key = '9eb9f6eac22bb9a13a90f36ab18092e455fef1ead93ec303fe94c0bf2ae90e42'
    bucket_name = 'tre'
    folder_local = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\jpg' ##os.getenv('CANDIDATO_FOTOS_DIR', '/opt/airflow/TRE2026/archives/RR/candidato/jpg')
    folder_remote = 'fotosrr/'               

    s3 = boto3.client(
    service_name='s3',
    endpoint_url=f'https://{account_id}.r2.cloudflarestorage.com',
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='auto', 
    config=Config(signature_version='s3v4')
    )
    base_url = "https://pub-e9ede5cdf96443d58340a9a3b62a2816.r2.dev/fotosrr/" 
    # engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")

    for i in os.listdir(file_csv):
        full_path = os.path.join(file_csv, i)
        if full_path.endswith('RR.csv'):
            print(f"Lendo arquivo: {full_path}")
            df = pd.read_csv(full_path, encoding='latin-1', sep=';',usecols=lambda col: col.strip().lower() in colunas_leitura,dtype=str)
            df.columns = [col.strip().lower() for col in df.columns]
            df['sg_ue'] = df['sg_ue'].astype(str).str.lstrip("0")
            df.loc[df['nm_ue'] == "RORAIMA", 'sg_ue'] = '3018'
            df['nm_candidato'] = (
                df['nm_urna_candidato'] 
                .fillna('')
                .astype(str)
                .str.normalize('NFKD')
                .str.encode('ascii', 'ignore')
                .str.decode('utf-8')
                .str.replace(r'[^A-Za-z0-9]+', '', regex=True)
            )
            df['id_candidato'] = df['ano_eleicao'] + df['nr_turno'] +  df['cd_cargo'] + df['sg_ue'] +  df['nr_candidato'] + df['nm_candidato']
                  
            dfs.append(df)


    dfconsulta = pd.concat(dfs, ignore_index=True)

    for file in os.listdir(folder_local):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            nome_tratado = file.replace("FRR", "RR").replace("_div", "")
            lista_links.append({
                "Nome_Foto": file,
                "URL_Foto": base_url + nome_tratado
            })
    fotos = pd.DataFrame(lista_links)
    # fotos['URL_Foto'] = fotos['URL_Foto'].str[:15]
    fotos['idCandidatos'] = fotos['Nome_Foto'].str[3:15]

# https://pub-e9ede5cdf96443d58340a9a3b62a2816.r2.dev/fotosrr/FRR230000632353_div.jpg
    df = pd.merge(
    dfconsulta, 
    fotos, 
    how='left', 
    right_on='idCandidatos',
    left_on='sq_candidato').drop('idCandidatos', axis=1) 
    df.drop_duplicates(['id_candidato'], inplace=True)

    # df.to_sql("dcandidato", engine, index=False, if_exists='append')


# FRR230000632353_div.jpg


    return df
df = candidato(file_csv = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv')
df

Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2020_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2022_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2024_RR.csv


,dt_geracao,hh_geracao,ano_eleicao,cd_tipo_eleicao,nm_tipo_eleicao,nr_turno,cd_eleicao,ds_eleicao,dt_eleicao,tp_abrangencia,...,ds_estado_civil,cd_cor_raca,ds_cor_raca,cd_ocupacao,ds_ocupacao,cd_sit_tot_turno,ds_sit_tot_turno,id_candidato,Nome_Foto,URL_Foto
0,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,DIVORCIADO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622333DANIELEDECAMPOSNOVOS,FRR230000726928_div.jpg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
1,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622222CASSIODOTAXI,FRR230000726931_div.jpg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
2,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),05,INDÍGENA,601,AGRICULTOR,5,SUPLENTE,2020113302611611MARA,FRR230000726780_div.jpg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302610555TANDYLIMA,FRR230000726736_div.jpg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
4,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,CASADO(A),03,PARDA,266,PROFESSOR DE ENSINO MÉDIO,5,SUPLENTE,2020113302610456PROFESSORAGNALDO,FRR230000726732_div.jpg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3845,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,999,OUTROS,5,SUPLENTE,2024113305010000RONYPETERSON,FRR230002064378_div.jpg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3846,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,2,ELEITO POR QP,2024113305010777SANDRIELYCUNHA,FRR230002390636_div.jpeg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3847,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,931,"ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS",5,SUPLENTE,2024113305020789MOACYPEDEMINGAL,FRR230002185648_div.jpeg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3848,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,CASADO(A),03,PARDA,257,EMPRESÁRIO,5,SUPLENTE,2024113305020333CHEILADOAUAU,FRR230002185651_div.jpeg,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...


Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2020_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2022_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2024_RR.csv


,dt_geracao,hh_geracao,ano_eleicao,cd_tipo_eleicao,nm_tipo_eleicao,nr_turno,cd_eleicao,ds_eleicao,dt_eleicao,tp_abrangencia,...,ds_estado_civil,cd_cor_raca,ds_cor_raca,cd_ocupacao,ds_ocupacao,cd_sit_tot_turno,ds_sit_tot_turno,id_candidato,Nome_Foto,URL_Foto
0,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,DIVORCIADO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622333DANIELEDECAMPOSNOVOS,RR230000726928,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
1,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622222CASSIODOTAXI,RR230000726931,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
2,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),05,INDÍGENA,601,AGRICULTOR,5,SUPLENTE,2020113302611611MARA,RR230000726780,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302610555TANDYLIMA,RR230000726736,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
4,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,CASADO(A),03,PARDA,266,PROFESSOR DE ENSINO MÉDIO,5,SUPLENTE,2020113302610456PROFESSORAGNALDO,RR230000726732,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3845,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,999,OUTROS,5,SUPLENTE,2024113305010000RONYPETERSON,RR230002064378,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3846,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,2,ELEITO POR QP,2024113305010777SANDRIELYCUNHA,RR230002390636,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3847,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,931,"ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS",5,SUPLENTE,2024113305020789MOACYPEDEMINGAL,RR230002185648,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...
3848,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,CASADO(A),03,PARDA,257,EMPRESÁRIO,5,SUPLENTE,2024113305020333CHEILADOAUAU,RR230002185651,https://pub-e9ede5cdf96443d58340a9a3b62a2816.r...


In [3]:
df = candidato(file_csv = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv')


Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2020_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2022_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2024_RR.csv


In [13]:
# df[df['nm_candidato'] == 'CLAUDIO']
df

,dt_geracao,hh_geracao,ano_eleicao,cd_tipo_eleicao,nm_tipo_eleicao,nr_turno,cd_eleicao,ds_eleicao,dt_eleicao,tp_abrangencia,...,ds_estado_civil,cd_cor_raca,ds_cor_raca,cd_ocupacao,ds_ocupacao,cd_sit_tot_turno,ds_sit_tot_turno,id_candidato,Nome_Foto,URL_Foto
0,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,DIVORCIADO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622333DANIELEDECAMPOSNOVOS,NaN,NaN
1,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622222CASSIODOTAXI,NaN,NaN
2,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),05,INDÍGENA,601,AGRICULTOR,5,SUPLENTE,2020113302611611MARA,NaN,NaN
3,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302610555TANDYLIMA,NaN,NaN
4,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,CASADO(A),03,PARDA,266,PROFESSOR DE ENSINO MÉDIO,5,SUPLENTE,2020113302610456PROFESSORAGNALDO,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3845,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,999,OUTROS,5,SUPLENTE,2024113305010000RONYPETERSON,NaN,NaN
3846,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),03,PARDA,999,OUTROS,2,ELEITO POR QP,2024113305010777SANDRIELYCUNHA,NaN,NaN
3847,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SOLTEIRO(A),01,BRANCA,931,"ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS",5,SUPLENTE,2024113305020789MOACYPEDEMINGAL,NaN,NaN
3848,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,CASADO(A),03,PARDA,257,EMPRESÁRIO,5,SUPLENTE,2024113305020333CHEILADOAUAU,NaN,NaN
